In [8]:
import pandas as pd

# Path to your CSV file
file_path = "df_FN.csv"  # ← change this to your actual filename

# 1. Load the dataset
df = pd.read_csv(file_path)

# 2. Define the desired column order by importance
ordered_cols_by_importance = [
    "Contract",
    "OnlineSecurity",
    "TechSupport",
    "InternetService",
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "PaperlessBilling",
    "StreamingTV",
    "OnlineBackup",
    "MultipleLines",
    "PaymentMethod",
    "PhoneService",
    "StreamingMovies",
    "Dependents",
    "SeniorCitizen",
    "customerID",
    "Partner",
    "gender",
    "DeviceProtection"
]

# 3. Keep only the columns that actually exist in your DataFrame
existing_ordered_cols = [col for col in ordered_cols_by_importance if col in df.columns]

# 4. Identify any remaining columns not listed above
remaining_cols = [col for col in df.columns if col not in existing_ordered_cols]

# 5. Create the final column list: first your important columns, then the rest
final_columns = existing_ordered_cols + remaining_cols

# 6. Reorder df accordingly
df = df[final_columns]

# 7. Overwrite the original CSV with this reordered DataFrame
df.to_csv(file_path, index=False)

print(f"Columns have been reordered and saved back to: {file_path}")


FileNotFoundError: [Errno 2] No such file or directory: 'df_FN.csv'

# LightGBM Hyperparameter Tuning and Cross-Validation

This notebook demonstrates hyperparameter tuning using Optuna and cross-validation for a LightGBM model.

In [ ]:
import pandas as pd
import numpy as np
import logging
from typing import Tuple, Dict, Any, Optional
from dataclasses import dataclass
from pathlib import Path
import joblib
import json

from lightgbm import LGBMClassifier, early_stopping
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    RandomizedSearchCV
)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    roc_curve,
    f1_score,
    accuracy_score
)
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

import ray

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

## Define Model Configuration

In [ ]:
@dataclass
class ModelConfig:
    """Configuration class for model hyperparameters and training settings."""
    test_size: float = 0.2
    random_state: int = 42
    cv_folds: int = 5
    n_jobs: int = -1
    early_stopping_rounds: int = 50

    # XGBoost hyperparameters for initial training
    base_params: Dict[str, Any] = None

    def __post_init__(self):
        if self.base_params is None:
            self.base_params = {
                'objective': 'binary',
                'random_state': self.random_state,
                'n_jobs': self.n_jobs,
                'verbosity': -1
            }

## Define ProductionLightGBMPipeline Class

In [ ]:
class ProductionLightGBMPipeline:
    """
    Production-grade LightGBM pipeline with proper early stopping handling.
    """

    def __init__(self, config: ModelConfig = None):
        self.config = config or ModelConfig()
        self.model = None
        self.feature_names = None
        self.label_encoders = {}
        self.model_metadata = {}
        self.performance_metrics = {}

    def preprocess_features(self, df: pd.DataFrame, is_training: bool = True) -> pd.DataFrame:
        """
        Advanced preprocessing with categorical encoding optimized for LightGBM compatibility.
        """
        logger.info("Starting feature preprocessing...")

        df_processed = df.copy()

        # Handle categorical variables with label encoding for LightGBM compatibility
        categorical_cols = df_processed.select_dtypes(include=['object', 'category']).columns
        categorical_cols = categorical_cols.drop('Churn', errors='ignore')  # Exclude target

        for col in categorical_cols:
            logger.info(f"Processing categorical column: {col}")

            if is_training:
                # Fit new encoder during training phase
                le = LabelEncoder()
                # Handle NaN values explicitly with proper masking
                mask = df_processed[col].notna()

                if mask.sum() > 0:  # Only proceed if non-null values exist
                    # Fit encoder on non-null values
                    unique_values = df_processed.loc[mask, col].unique()
                    le.fit(unique_values)
                    self.label_encoders[col] = le

                    # Transform non-null values
                    df_processed.loc[mask, col] = le.transform(df_processed.loc[mask, col])

                    # Convert to nullable integer type for LightGBM compatibility
                    df_processed[col] = df_processed[col].astype('Int64')  # Pandas nullable integer

                    logger.info(f"Encoded {col}: {len(unique_values)} unique values → integers")
                else:
                    logger.warning(f"Column {col} contains only NaN values")
                    df_processed[col] = pd.Series(dtype='Int64')  # Empty nullable integer series

            else:
                # Apply existing encoder during inference phase
                if col in self.label_encoders:
                    le = self.label_encoders[col]
                    mask = df_processed[col].notna()

                    if mask.sum() > 0:
                        # Handle unseen categories with robust fallback strategy
                        seen_values = set(le.classes_)
                        current_values = set(df_processed.loc[mask, col].unique())
                        unseen_values = current_values - seen_values

                        if unseen_values:
                            logger.warning(f"Unseen categories in {col}: {unseen_values}")
                            # Map unseen categories to most frequent encoded value (0)
                            most_frequent_encoded = 0
                            unseen_mask = mask & df_processed[col].isin(unseen_values)
                            df_processed.loc[unseen_mask, col] = most_frequent_encoded

                        # Transform known categories
                        known_mask = mask & df_processed[col].isin(seen_values)
                        if known_mask.sum() > 0:
                            df_processed.loc[known_mask, col] = le.transform(df_processed.loc[known_mask, col])

                    # Ensure consistent dtype
                    df_processed[col] = df_processed[col].astype('Int64')
                else:
                    logger.warning(f"No encoder found for column {col} during inference")
                    df_processed[col] = pd.Series(dtype='Int64')

        # Validate final dtypes for LightGBM compatibility
        invalid_dtypes = []
        for col in df_processed.columns:
            if col == 'Churn':  # Skip target column
                continue
            dtype = df_processed[col].dtype
            if not (pd.api.types.is_numeric_dtype(dtype) or pd.api.types.is_bool_dtype(dtype)):
                invalid_dtypes.append(f"{col}: {dtype}")

        if invalid_dtypes:
            logger.error(f"Invalid dtypes detected: {invalid_dtypes}")
            raise ValueError(f"LightGBM incompatible dtypes found: {invalid_dtypes}")

        logger.info(f"Preprocessing complete. Shape: {df_processed.shape}")
        logger.info(f"Final dtypes: {df_processed.dtypes.value_counts().to_dict()}")
        logger.info(f"Total NaN values: {df_processed.isnull().sum().sum()}")

        return df_processed

    def optimize_hyperparameters(self, X_train: pd.DataFrame, y_train: pd.Series) -> Dict[str, Any]:
        """
        Systematic hyperparameter optimization with Ray for parallelism and logging for each fit.
        """
        logger.info("Starting hyperparameter optimization...")
        pos_samples = (y_train == 1).sum()
        neg_samples = (y_train == 0).sum()
        scale_pos_weight = neg_samples / pos_samples if pos_samples > 0 else 1
        param_grid = {
            'num_leaves': [15, 31, 63, 127],
            'max_depth': [3, 5, 7, 10, -1],
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'n_estimators': [200, 400, 600, 1000],
            'min_child_samples': [10, 20, 30, 50],
            'subsample': [0.7, 0.8, 0.9, 1.0],
            'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
            'scale_pos_weight': [1, scale_pos_weight]
        }
        cv_strategy = StratifiedKFold(
            n_splits=5,  # Increased to 5 folds
            shuffle=True,
            random_state=self.config.random_state
        )
        base_model = LGBMClassifier(**self.config.base_params)
        # Initialize Ray
        if not ray.is_initialized():
            ray.init(ignore_reinit_error=True, log_to_driver=False)
        search = RandomizedSearchCV(
            estimator=base_model,
            param_distributions=param_grid,
            n_iter=20,  # 20 candidates
            cv=cv_strategy,
            scoring='roc_auc',
            n_jobs=-1,  # Use all available cores
            verbose=1,
            random_state=self.config.random_state
        )
        try:
            search.fit(X_train, y_train, 
                       callbacks=[lambda x: logger.info(f"Completed fit: {x}")])
            best_params = search.best_params_
            best_score = search.best_score_
            logger.info(f"Best CV AUC Score: {best_score:.4f}")
            logger.info(f"Best Parameters: {best_params}")
            return best_params
        except Exception as e:
            logger.error(f"Hyperparameter optimization failed: {str(e)}")
            return {
                'n_estimators': 400,
                'max_depth': 6,
                'learning_rate': 0.1,
                'subsample': 0.8,
                'colsample_bytree': 0.8,
                'min_child_samples': 20,
                'num_leaves': 31,
                'scale_pos_weight': scale_pos_weight
            }

    def train_model(self, df: pd.DataFrame, target_col: str = 'Churn') -> Dict[str, Any]:
        """
        Complete training pipeline with proper early stopping implementation.
        """
        logger.info("Starting model training pipeline...")

        # Data validation
        if df.empty:
            raise ValueError("Input dataframe is empty")

        if target_col not in df.columns:
            raise ValueError(f"Target column '{target_col}' not found in dataframe")

        # Preprocess features
        df_processed = self.preprocess_features(df, is_training=True)

        # Separate features and target
        X = df_processed.drop(columns=[target_col])
        y = df_processed[target_col]

        # Encode target variable to binary integers if string-based
        if y.dtype == 'object' or y.dtype.name == 'category':
            logger.info(f"Encoding target variable. Original values: {y.unique()}")
            target_encoder = LabelEncoder()
            y_encoded = target_encoder.fit_transform(y)
            y = pd.Series(y_encoded, index=y.index, name=y.name)
            self.model_metadata['target_encoder'] = target_encoder
            self.model_metadata['target_mapping'] = dict(zip(target_encoder.classes_,
                                                           target_encoder.transform(target_encoder.classes_)))
            logger.info(f"Target encoding mapping: {self.model_metadata['target_mapping']}")
        else:
            self.model_metadata['target_encoder'] = None

        # Store feature names for deployment
        self.feature_names = X.columns.tolist()

        logger.info(f"Dataset shape: {X.shape}")
        logger.info(f"Class distribution: {y.value_counts().to_dict()}")

        # Three-way stratified split: train/validation/test
        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y,
            test_size=self.config.test_size,
            random_state=self.config.random_state,
            stratify=y
        )

        validation_ratio = 0.2
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp,
            test_size=validation_ratio,
            random_state=self.config.random_state,
            stratify=y_temp
        )

        logger.info(f"Data splits - Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

        # Optimize hyperparameters using only training data
        best_params = self.optimize_hyperparameters(X_train, y_train)

        # Merge with base parameters for final training
        final_params = {**self.config.base_params, **best_params}

        # Train final model with early stopping using validation set
        final_params['early_stopping_rounds'] = self.config.early_stopping_rounds
        self.model = LGBMClassifier(**final_params)

        # Fit with explicit validation set for early stopping
        self.model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            eval_metric='auc',
            callbacks=[early_stopping(self.config.early_stopping_rounds)],
            verbose=False
        )

        # Generate comprehensive evaluation
        train_results = self._evaluate_model(X_train, X_val, X_test, y_train, y_val, y_test)

        # Store model metadata
        self.model_metadata.update({
            'feature_names': self.feature_names,
            'model_params': final_params,
            'training_shape': X_train.shape,
            'class_distribution': y_train.value_counts().to_dict(),
            'best_iteration': getattr(self.model, 'best_iteration_', None)
        })

        logger.info("Model training completed successfully")
        
        # Print formatted results
        print_formatted_results(train_results)
        
        return train_results

    def _evaluate_model(self, X_train: pd.DataFrame, X_val: pd.DataFrame, X_test: pd.DataFrame,
                       y_train: pd.Series, y_val: pd.Series, y_test: pd.Series) -> Dict[str, Any]:
        """Comprehensive model evaluation across train/validation/test splits."""

        # Generate predictions across all data partitions
        y_train_pred = self.model.predict(X_train)
        y_val_pred = self.model.predict(X_val)
        y_test_pred = self.model.predict(X_test)

        y_train_proba = self.model.predict_proba(X_train)[:, 1]
        y_val_proba = self.model.predict_proba(X_val)[:, 1]
        y_test_proba = self.model.predict_proba(X_test)[:, 1]

        # Calculate comprehensive metrics
        metrics = {
            'train_accuracy': accuracy_score(y_train, y_train_pred),
            'train_auc': roc_auc_score(y_train, y_train_proba),
            'train_f1': f1_score(y_train, y_train_pred),

            'val_accuracy': accuracy_score(y_val, y_val_pred),
            'val_auc': roc_auc_score(y_val, y_val_proba),
            'val_f1': f1_score(y_val, y_val_pred),

            'test_accuracy': accuracy_score(y_test, y_test_pred),
            'test_auc': roc_auc_score(y_test, y_test_proba),
            'test_f1': f1_score(y_test, y_test_pred),

            'confusion_matrix_test': confusion_matrix(y_test, y_test_pred),
            'classification_report': classification_report(y_test, y_test_pred, output_dict=True)
        }

        # Cross-validation for robust performance estimation
        # Use a clean LightGBM model for CV (without early stopping)
        cv_model = LGBMClassifier(**{k: v for k, v in self.model.get_params().items()
                                   if k != 'early_stopping_rounds'})

        cv_scores = cross_val_score(
            cv_model, X_train, y_train,
            cv=StratifiedKFold(n_splits=self.config.cv_folds, shuffle=True, random_state=self.config.random_state),
            scoring='roc_auc',
            n_jobs=self.config.n_jobs
        )

        metrics['cv_auc_mean'] = cv_scores.mean()
        metrics['cv_auc_std'] = cv_scores.std()

        # Feature importance analysis
        if hasattr(self.model, 'feature_importances_'):
            feature_importance = pd.DataFrame({
                'feature': self.feature_names,
                'importance': self.model.feature_importances_
            }).sort_values('importance', ascending=False)

            metrics['feature_importance'] = feature_importance

        # Early stopping diagnostics
        if hasattr(self.model, 'best_iteration_'):
            metrics['best_iteration'] = self.model.best_iteration_
            metrics['total_boosting_rounds'] = self.model.n_estimators
            metrics['early_stopping_triggered'] = self.model.best_iteration_ < self.model.n_estimators

        self.performance_metrics = metrics

        # Log performance summary (keep existing logging for debugging)
        logger.info("=== Model Performance Summary ===")
        logger.info(f"Train AUC: {metrics['train_auc']:.4f} | Val AUC: {metrics['val_auc']:.4f} | Test AUC: {metrics['test_auc']:.4f}")
        logger.info(f"Train F1:  {metrics['train_f1']:.4f} | Val F1:  {metrics['val_f1']:.4f} | Test F1:  {metrics['test_f1']:.4f}")
        logger.info(f"CV AUC: {metrics['cv_auc_mean']:.4f} (+/- {metrics['cv_auc_std']*2:.4f})")

        if 'early_stopping_triggered' in metrics:
            logger.info(f"Early stopping: {'Yes' if metrics['early_stopping_triggered'] else 'No'} "
                       f"(stopped at {metrics.get('best_iteration', 'N/A')}/{metrics.get('total_boosting_rounds', 'N/A')})")

        return metrics

    def predict(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        """Production-ready prediction method with comprehensive validation."""
        if self.model is None:
            raise ValueError("Model has not been trained. Call train_model() first.")

        # Preprocess features using training encoders
        df_processed = self.preprocess_features(df, is_training=False)

        # Handle target column if present in prediction data
        if 'Churn' in df_processed.columns:
            df_processed = df_processed.drop(columns=['Churn'])

        # Ensure feature consistency
        missing_features = set(self.feature_names) - set(df_processed.columns)
        if missing_features:
            logger.warning(f"Missing features in prediction data: {missing_features}")
            for feature in missing_features:
                df_processed[feature] = np.nan

        # Select and order features to match training
        X_pred = df_processed[self.feature_names]

        # Generate predictions
        predictions = self.model.predict(X_pred)
        probabilities = self.model.predict_proba(X_pred)[:, 1]

        # Decode predictions back to original labels if target encoder exists
        if 'target_encoder' in self.model_metadata and self.model_metadata['target_encoder'] is not None:
            predictions = self.model_metadata['target_encoder'].inverse_transform(predictions)

        return predictions, probabilities

    def save_model(self, filepath: str) -> None:
        """Save complete model pipeline for deployment."""
        if self.model is None:
            raise ValueError("No trained model to save")

        model_package = {
            'model': self.model,
            'label_encoders': self.label_encoders,
            'feature_names': self.feature_names,
            'metadata': self.model_metadata,
            'config': self.config
        }

        joblib.dump(model_package, filepath)
        logger.info(f"Model saved to {filepath}")

        # Save performance metrics separately
        metrics_path = Path(filepath).with_suffix('.json')
        with open(metrics_path, 'w') as f:
            serializable_metrics = {}
            for key, value in self.performance_metrics.items():
                if isinstance(value, np.ndarray):
                    serializable_metrics[key] = value.tolist()
                elif isinstance(value, pd.DataFrame):
                    serializable_metrics[key] = value.to_dict()
                else:
                    serializable_metrics[key] = value

            json.dump(serializable_metrics, f, indent=2, default=str)

    @classmethod
    def load_model(cls, filepath: str) -> 'ProductionLightGBMPipeline':
        """Load complete model pipeline from file."""
        model_package = joblib.load(filepath)

        pipeline = cls(config=model_package['config'])
        pipeline.model = model_package['model']
        pipeline.label_encoders = model_package['label_encoders']
        pipeline.feature_names = model_package['feature_names']
        pipeline.model_metadata = model_package['metadata']

        logger.info(f"Model loaded from {filepath}")
        return pipeline

    def run_cross_validation(self, X: pd.DataFrame, y: pd.Series) -> Tuple[float, float]:
        """Run cross-validation on the entire dataset using the best hyperparameters."""
        best_params = self.optimize_hyperparameters(X, y)
        model = LGBMClassifier(**best_params)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=self.config.random_state)  # 5 folds
        scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
        logger.info(f"Cross-validation completed: mean={scores.mean():.4f}, std={scores.std():.4f}")
        return scores.mean(), scores.std()

## Main Execution Block

In [ ]:
def main():
    """Example usage of the production pipeline with output matching the provided screenshot format."""
    config = ModelConfig(
        test_size=0.2,
        cv_folds=5,
        early_stopping_rounds=50
    )

    pipeline = ProductionLightGBMPipeline(config)

    # Assuming df_xgb is your dataset
    results = pipeline.train_model(df_xgb)

    # Run cross-validation on the entire dataset
    X = df_xgb.drop(columns=['Churn'])
    y = df_xgb['Churn']
    cv_mean, cv_std = pipeline.run_cross_validation(X, y)
    print(f"Cross-Validation AUC: {cv_mean:.4f} (+/- {cv_std*2:.4f})")

    # Optuna tuning removed
    # Output in the required format (already handled by print_formatted_results in train_model)
    pass

if __name__ == "__main__":
    main()